# Monitoring a deployed model: windows, drift and alerts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkcs28/evalsuite-python/blob/main/notebooks/04_monitoring_drift.ipynb)

`Monitor` recomputes metrics over a sliding window as labels arrive, compares scores with the validation distribution (PSI, Kolmogorov–Smirnov) and fires alerts.

In [ ]:
%pip install -q evalsuite-python scikit-learn

In [ ]:
import numpy as np
import evalsuite as es

rng = np.random.default_rng(1)
def batch(n, shift=0.0):
    y = (rng.random(n) < 0.4).astype(int)
    p = np.clip(np.where(y == 1, 0.7 - shift, 0.3 + shift) + rng.normal(0, 0.15, n), 0.01, 0.99)
    return y, p

_, validation_scores = batch(2000)
mon = es.Monitor(window=500, alerts={"roc_auc": ("<", 0.8), "psi": (">", 0.25)},
                 reference_scores=validation_scores, min_n=200)

In [ ]:
for week, shift in enumerate([0.0, 0.0, 0.05, 0.12, 0.2], start=1):
    y, p = batch(250, shift)
    fired = mon.update(y, y_pred=(p >= 0.5).astype(int), y_prob=p)
    print(f"week {week}:", fired or "ok")

In [ ]:
print(mon.snapshot().summary())
print(mon.drift())